In [133]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pointbiserialr

pio.templates.default = "plotly_dark"

In [134]:
df_healthy = pd.read_parquet("../../data/processed/dataset_features.parquet")

print(df_healthy["age"].value_counts())
print(df_healthy["gender"].value_counts())
print(df_healthy["handedness"].value_counts())
print(df_healthy["education"].value_counts())
print(df_healthy["drug"].value_counts())

# garantir tipos
df_healthy["age"] = df_healthy["age"].astype(str)
df_healthy["gender"] = df_healthy["gender"].astype(str)
df_healthy["handedness"] = df_healthy["handedness"].astype(str)
df_healthy["education"] = df_healthy["education"].astype(str)
df_healthy["drug"] = df_healthy["drug"].astype(str)


age
25-30    24
20-25    16
65-70     4
30-35     4
60-65     2
55-60     2
70-75     2
Name: count, dtype: int64
gender
2    38
1    16
Name: count, dtype: int64
handedness
right          50
left            2
ambedextor      2
Name: count, dtype: int64
education
gymnasium     40
realschule    12
gymansium      2
Name: count, dtype: int64
drug
0.0    54
Name: count, dtype: int64


In [135]:
# Organizando as variáveis
df_healthy["age_group"] = df_healthy["age"].replace({
    "20-25": "young",
    "25-30": "young",
    "30-35": "young",
    "35-40": "young",
    "55-60": "older",
    "60-65": "older",
    "65-70": "older",
    "70-75": "older"
})

# Calculando as médias das métricas
degree_cols = [c for c in df_healthy.columns if c.startswith("degree_")]
clustering_cols = [c for c in df_healthy.columns if c.startswith("clustering_")]
efficiency_cols = [c for c in df_healthy.columns if c.startswith("efficiency_")]
betweenness_cols = [c for c in df_healthy.columns if c.startswith("betweenness_")]
df_healthy["degree_mean"] = df_healthy[degree_cols].mean(axis=1)
df_healthy["clustering_mean"] = df_healthy[clustering_cols].mean(axis=1)
df_healthy["efficiency_global"] = df_healthy[efficiency_cols].mean(axis=1)
df_healthy["betweenness_mean"] = df_healthy[betweenness_cols].mean(axis=1)


In [136]:

GRAPH_METRICS = {
    "degree_mean": "Grau Médio",
    "clustering_mean": "Coeficiente de Clustering Médio",
    "betweenness_mean": "Centralidade de Intermediação Médio"
}

sample_summary = (
    df_healthy[["subject_id", "age_group", "gender"]]
    .drop_duplicates()
    .groupby(["age_group", "gender"])
    .size()
    .reset_index(name="n")
    
)

sample_summary

,age_group,gender,n
0,older,1,2
1,older,2,3
2,young,1,6
3,young,2,16


In [137]:
df_healthy = (
    df_healthy[df_healthy["condition"] != "E0"]
    .rename(columns={"age_group": "Faixa Etária", "gender": "Gênero"})
    .copy()
)

df_healthy[["Faixa Etária", "Gênero"]].value_counts()

Faixa Etária  Gênero
young         2         31
              1         12
older         2          6
              1          4
Name: count, dtype: int64

### Idade

In [139]:
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="Faixa Etária",
        title=f"{label} vs Age Group",
        points='all',
        labels={
            "Faixa Etária": "Faixa Etária",
            metric: label
        }
    )
    fig.update_layout(
        xaxis_title="Faixa Etária",
        yaxis_title=label,
        title=f"{label} vs Faixa Etária"
    )
    fig.show()

-------

### Condition

In [ ]:

# plotly
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="condition",
        y=metric,
        color="condition",
        title=f"{metric} vs Condition",
        points="all",
        labels={
            "condition": "Condition",
            metric: metric
        }
    )
    fig.update_layout(
        xaxis_title="Condition",
        yaxis_title=label,
        title=f"{label} vs Condition"
    )
    fig.show()

------

In [144]:
# Boxplot Age Group x Condition
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="condition",
        title=f"{label} vs Faixa Etária",
        points="all",
        labels={
            "Faixa Etária": "Faixa Etária",
            label: label
        }
    )
    fig.update_layout(
        xaxis_title=f"{label}: EO vs EC estratificado por idade",
        yaxis_title=label,
        title=f"{label} vs Faixa Etária"
    )
    fig.show()

------------------

### Correlações

In [ ]:
# Correlation Age Group
age_ord_map = {"young": 0, "middle": 1, "older": 2}
df_healthy["age_ord"] = df_healthy["age_group"].map(age_ord_map)

corr_results = []

for metric in GRAPH_METRICS:
    valid = df_healthy[[metric, "age_ord"]].dropna()
    r, p = spearmanr(valid["age_ord"], valid[metric])
    corr_results.append({
        "metric": metric,
        "spearman_r": r,
        "p_value": p
    })

pd.DataFrame(corr_results)

,metric,spearman_r,p_value
0,degree_mean,-0.267964,0.052388
1,clustering_mean,-0.308947,0.024391
2,betweenness_mean,0.377756,0.005292


As redes funcionais tendem a apresentar menor conectividade média em indivíduos mais velhos e uma possível redução da segregação local das redes funcionais com o envelhecimento. Com o envelhecimento, a comunicação funcional pode se tornar mais dependente de nós específicos, indicando possível perda de redundância global.

Uma interpretação complementar para os resultados observados é que o envelhecimento esteja associado a uma progressiva esparsificação das redes funcionais cerebrais. A redução do grau médio e do coeficiente de agrupamento sugere uma diminuição tanto da densidade global quanto da conectividade local das redes. Como consequência, a comunicação funcional tende a se apoiar em um número menor de nós intermediários, o que se reflete no aumento da centralidade de intermediação média. Esse padrão indica uma possível perda de redundância funcional e maior vulnerabilidade da rede, sendo consistente com modelos de envelhecimento cerebral descritos na literatura.

Gênero

In [ ]:
df_healthy["gender"].value_counts()

gender
2    37
1    16
Name: count, dtype: int64

In [ ]:
df_healthy["gender"].dtype
df_healthy["gender"].unique()


array(['2', '1'], dtype=object)

In [ ]:
df_healthy["gender"] = df_healthy["gender"].astype(float).astype(int)

df_healthy["gender_bin"] = df_healthy["gender"].map({
    1: 0,  # female
    2: 1   # male
})

df_healthy['gender_bin'].value_counts()


gender_bin
1    37
0    16
Name: count, dtype: int64

In [ ]:
gender_results = []

for metric in GRAPH_METRICS:
    valid = df_healthy[[metric, "gender_bin"]].dropna()
    r, p = pointbiserialr(valid["gender_bin"], valid[metric])
    gender_results.append({
        "metric": metric,
        "point_biserial_r": r,
        "p_value": p
    })

pd.DataFrame(gender_results)



,metric,point_biserial_r,p_value
0,degree_mean,0.035857,0.798799
1,clustering_mean,0.005495,0.968851
2,betweenness_mean,-0.182980,0.189705


Idade

In [ ]:
age_ord_map = {"young": 0, "older": 1}
df_healthy["age_ord"] = df_healthy["age_group"].map(age_ord_map)

corr_results = []

for metric in GRAPH_METRICS:
    valid = df_healthy[[metric, "age_ord"]].dropna()
    r, p = spearmanr(valid["age_ord"], valid[metric])
    corr_results.append({
        "metric": metric,
        "spearman_r": r,
        "p_value": p
    })

pd.DataFrame(corr_results)

,metric,spearman_r,p_value
0,degree_mean,-0.267964,0.052388
1,clustering_mean,-0.308947,0.024391
2,betweenness_mean,0.377756,0.005292


Os resultados indicam que, mesmo em indivíduos saudáveis, o envelhecimento está associado a uma reorganização topológica global das redes funcionais, com redução da densidade e modularidade e aumento de graus intermediários.